In [ ]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

In [ ]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile 
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math 

# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.


# BB84 with an attacker

This notebook simulates BB84 when Eve performs an intercept-resend attack. The code is still one sequential program, but comments label Alice, Eve, and Bob.

No Python standard random generator is used. Alice's bits, Alice's bases, Bob's bases, and Eve's bases all come from measuring qubits prepared as `1/sqrt(2)(|0> + |1>)`.

In [ ]:
backend = BasicSimulator()


def run_circuit_once(circuit):
    """Run a circuit once and return the measured bit string."""
    compiled = transpile(circuit, backend)
    result = backend.run(compiled, shots=1).result()
    return next(iter(result.get_counts()))


def quantum_random_bits(number_of_bits):
    """Generate random bits by measuring Hadamard-prepared qubits."""
    bits = []
    batch_size = 16

    while len(bits) < number_of_bits:
        current_batch_size = min(batch_size, number_of_bits - len(bits))
        circuit = QuantumCircuit(current_batch_size, current_batch_size)

        # Each qubit starts in |0>. Applying H gives 1/sqrt(2)(|0> + |1>).
        for qubit in range(current_batch_size):
            circuit.h(qubit)
            circuit.measure(qubit, qubit)

        bit_string = run_circuit_once(circuit)
        bits.extend(int(bit) for bit in bit_string)

    return bits


def prepare_qubit(bit, basis):
    """Prepare one BB84 qubit.

    basis 0 = computational/Z basis: |0>, |1>
    basis 1 = diagonal/X basis: |+>, |->
    """
    circuit = QuantumCircuit(1, 1)

    if bit == 1:
        circuit.x(0)
    if basis == 1:
        circuit.h(0)

    return circuit


def measure_qubit(circuit, basis):
    """Measure one BB84 qubit in the selected basis."""
    if basis == 1:
        circuit.h(0)

    circuit.measure(0, 0)
    return int(run_circuit_once(circuit))


## Sequential BB84 program with Eve

Eve does not know Alice's bases, so she chooses her own random basis, measures each travelling qubit, and sends Bob a new qubit matching her measurement result. If Eve used the wrong basis, this can disturb the state. Alice and Bob detect this by revealing a sample of their sifted key and checking whether the proportion of mismatches is above a threshold.

In [ ]:
# -----------------------------
# Protocol settings
# -----------------------------
NUMBER_OF_QUBITS = 300
TEST_SAMPLE_SIZE = 100
DISRUPTION_THRESHOLD = 0.12


# -----------------------------
# Alice: choose bits and bases
# -----------------------------
alice_bits = quantum_random_bits(NUMBER_OF_QUBITS)
alice_bases = quantum_random_bits(NUMBER_OF_QUBITS)


# -----------------------------
# Bob: choose measurement bases
# -----------------------------
bob_bases = quantum_random_bits(NUMBER_OF_QUBITS)
bob_results = []


# -----------------------------
# Eve: choose attack bases
# -----------------------------
# In this demonstration Eve intercepts every qubit. This makes the
# disruption clear while still using quantum randomness for her bases.
eve_attacks = [True for index in range(NUMBER_OF_QUBITS)]
eve_bases = quantum_random_bits(NUMBER_OF_QUBITS)
eve_results = []


# -----------------------------
# Quantum channel: Alice sends qubits, Eve may intercept, Bob measures
# -----------------------------
for index in range(NUMBER_OF_QUBITS):
    # Alice prepares a BB84 qubit.
    qubit = prepare_qubit(alice_bits[index], alice_bases[index])

    if eve_attacks[index]:
        # Eve measures the travelling qubit in her own random basis.
        eve_result = measure_qubit(qubit, eve_bases[index])
        eve_results.append(eve_result)

        # Eve resends a replacement qubit prepared from what she measured.
        qubit = prepare_qubit(eve_result, eve_bases[index])
    else:
        eve_results.append(None)

    # Bob measures the qubit he receives.
    bob_results.append(measure_qubit(qubit, bob_bases[index]))


# -----------------------------
# Alice and Bob: public basis comparison and sifting
# -----------------------------
matching_basis_positions = [
    index
    for index in range(NUMBER_OF_QUBITS)
    if alice_bases[index] == bob_bases[index]
]

test_positions = matching_basis_positions[:TEST_SAMPLE_SIZE]
key_positions = matching_basis_positions[TEST_SAMPLE_SIZE:]

test_errors = sum(
    1
    for index in test_positions
    if alice_bits[index] != bob_results[index]
)
disruption_rate = test_errors / len(test_positions) if test_positions else 0
attacker_detected = disruption_rate > DISRUPTION_THRESHOLD

alice_key = [alice_bits[index] for index in key_positions]
bob_key = [bob_results[index] for index in key_positions]


# -----------------------------
# Results
# -----------------------------
print("BB84 with intercept-resend attacker")
print("Number of transmitted qubits:", NUMBER_OF_QUBITS)
print("Number of qubits Eve attacked:", sum(eve_attacks))
print("Matching basis positions:", len(matching_basis_positions))
print("Test sample size:", len(test_positions))
print("Test errors:", test_errors)
print("Disruption rate:", disruption_rate)
print("Disruption threshold:", DISRUPTION_THRESHOLD)
print("Attacker detected:", attacker_detected)
print("Alice remaining key:", alice_key)
print("Bob remaining key:  ", bob_key)
print("Remaining keys match before error correction:", alice_key == bob_key)

if attacker_detected:
    print("Conclusion: Alice and Bob should reject this key.")
else:
    print("Conclusion: no attacker was detected in this run, but BB84 detection is statistical.")
